[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Fgram-devAI/nlp-ncsr-task3/blob/main/notebooks/08_q8_roberta.ipynb)

# Q8 — NER with RoBERTa-base (CoNLL-2003)

NCSR Athens, NLP Assignment 3, Question 8. Self-contained Colab notebook.

**Modification of `src/NER-BERT.py` (the assignment starter):**
- Pre-trained model swapped from `bert-base-uncased` (BERT, WordPiece) → [`FacebookAI/roberta-base`](https://huggingface.co/FacebookAI/roberta-base) (RoBERTa, byte-level BPE).
- Tokenizer initialised with `add_prefix_space=True` — **required** for `is_split_into_words=True` with RoBERTa BPE; without it the first subword of every non-first word is silently mis-tokenized and entity F1 collapses by ~10 absolute points (brainstorming spec risk R5).
- Model class swapped from `BertForTokenClassification` → `RobertaForTokenClassification`.
- Same NER task, same `ner_tags`, same 9 BIO classes, same hyperparameters, same 3 seeds `[42, 43, 44]`, same 4 metrics, same JSON schema as Q1 — only the `model` field changes from `"bert-base-uncased"` → `"roberta-base"`.
- Per-seed metrics JSON written to `RESULTS_DIR/seed_<seed>.json`.

**Runtime**: `Runtime → Change runtime type → T4 GPU`. RoBERTa-base is the same parameter count as BERT-base (~125M); expect ~10–15 min/seed on T4 — similar to Q1's MPS wall-time scaled to T4.

In [ ]:
# Silence Pylance/pyright stub-strictness for this Colab-only notebook.
# Stub issues from torch / transformers / sklearn / seqeval are not runtime bugs.
# pyright: reportArgumentType=false, reportAttributeAccessIssue=false, reportPrivateImportUsage=false, reportReturnType=false

## 1. Environment check

In [ ]:
!nvidia-smi || echo 'NO GPU — switch to a GPU runtime before continuing.'

## 2. Install dependencies
Colab ships `torch`, `transformers`, `sklearn`, `tqdm`. Q8 needs `seqeval` (entity metrics) and `kagglehub` (dataset download).

In [ ]:
%pip install -q seqeval 'kagglehub<1.0'

## 3. Kaggle credentials → environment
Set the secrets in Colab (key icon in left sidebar): `KAGGLE_USERNAME`, `KAGGLE_KEY`. Then run this cell once per session.

In [ ]:
import os
from google.colab import userdata

os.environ['KAGGLE_USERNAME'] = userdata.get('KAGGLE_USERNAME')
os.environ['KAGGLE_KEY'] = userdata.get('KAGGLE_KEY')
print('Kaggle user loaded:', os.environ['KAGGLE_USERNAME'])

## 4. Download CoNLL-2003

In [ ]:
import kagglehub
from pathlib import Path

dataset_path = Path(kagglehub.dataset_download('alaakhaled/conll003-englishversion'))

def _locate(stem: str) -> Path:
    hits = list(dataset_path.rglob(f'{stem}.txt'))
    if not hits:
        raise FileNotFoundError(
            f'Could not find {stem}.txt under {dataset_path}. '
            'The Kaggle dataset layout may have changed; inspect the directory listing.'
        )
    return hits[0]

train_file = _locate('train')
valid_file = _locate('valid')
test_file = _locate('test')
print('train:', train_file)
print('valid:', valid_file)
print('test :', test_file)

# Sanity-check the split sizes against the assignment preliminaries (PDF p.2):
# train=14,041, valid=3,250, test=3,453.
expected = {'train': 14041, 'valid': 3250, 'test': 3453}
for label, path in (('train', train_file), ('valid', valid_file), ('test', test_file)):
    sent_count = sum(1 for line in open(path) if line.startswith('-DOCSTART-') or not line.strip()) - 1
    print(f'{label}: {sent_count} sentence boundaries (expected ~{expected[label]})')

## 5. Imports + hyperparameters

In [ ]:
import json
import random
import statistics
import subprocess
import time

import numpy as np
import torch
import torch.optim as optim
import transformers
# Q8 CHANGE: swap BertForTokenClassification → RobertaForTokenClassification
from transformers import AutoTokenizer, RobertaForTokenClassification
from sklearn.metrics import accuracy_score, balanced_accuracy_score, classification_report
from seqeval.metrics import classification_report as seqeval_report
from seqeval.metrics import f1_score as seqeval_f1
from seqeval.metrics import precision_score as seqeval_precision
from seqeval.metrics import recall_score as seqeval_recall
from tqdm.auto import tqdm

# Pin a record of the runtime versions so the report can cite them.
print(f'torch        {torch.__version__}')
print(f'transformers {transformers.__version__}')
print(f'numpy        {np.__version__}')

# hyper-parameters (identical to Q1 for a fair comparison)
EPOCHS = 3
BATCH_SIZE = 8
LR = 1e-5
SEEDS = [42, 43, 44]

# device select: CUDA on Colab, MPS local, CPU fallback
if torch.cuda.is_available():
    device = torch.device('cuda')
elif torch.backends.mps.is_available():
    device = torch.device('mps')
else:
    device = torch.device('cpu')
print('device:', device)

# default results location — overridden by the next cell if Drive is mounted
RESULTS_DIR = Path('/content/results/q8')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

## 5b. (Optional) Persist results to Google Drive

Mount Drive so per-seed JSONs survive Colab runtime kills. If you skip this cell, results land at `/content/results/q8/` (ephemeral). If you run it, all writes downstream redirect to `MyDrive/nlp-ncsr-task3-results/q8/` automatically.

In [ ]:
# Mount Google Drive (first run shows an OAuth popup — accept once per session).
from google.colab import drive
drive.mount('/content/drive')

# Redirect RESULTS_DIR through Drive.
RESULTS_DIR = Path('/content/drive/MyDrive/nlp-ncsr-task3-results/q8')
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
print('per-seed JSONs will now be saved to:', RESULTS_DIR)

## 6. Load CoNLL-2003 sentences
Verbatim from `NER-BERT.py` — Q8 picks `ner_tags` downstream (same as Q1).

In [ ]:
def load_sentences(filepath):
    sentences = []
    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
    with open(filepath, 'r', encoding='utf-8') as f:
        for line in f.readlines():
            if line.startswith('-DOCSTART-') or line.strip() == '':
                if len(tokens) > 0:
                    sentences.append({
                        'tokens': tokens, 'pos_tags': pos_tags,
                        'chunk_tags': chunk_tags, 'ner_tags': ner_tags,
                    })
                    tokens, pos_tags, chunk_tags, ner_tags = [], [], [], []
            else:
                l = line.strip().split(' ')
                if len(l) >= 4:
                    tokens.append(l[0]); pos_tags.append(l[1])
                    chunk_tags.append(l[2]); ner_tags.append(l[3])
    if len(tokens) > 0:
        sentences.append({
            'tokens': tokens, 'pos_tags': pos_tags,
            'chunk_tags': chunk_tags, 'ner_tags': ner_tags,
        })
    return sentences

print('loading data')
train_sentences = load_sentences(train_file)
valid_sentences = load_sentences(valid_file)
test_sentences = load_sentences(test_file)
print(f'train={len(train_sentences)}, valid={len(valid_sentences)}, test={len(test_sentences)}')

## 7. Label vocabulary — NER tags (unchanged from Q1)

In [ ]:
all_tags = sorted({tag for s in train_sentences for tag in s['ner_tags']})
label2id = {tag: i for i, tag in enumerate(all_tags)}
id2label = {i: tag for tag, i in label2id.items()}
num_labels = len(all_tags)
print('Tagset size:', num_labels)
print('Tags:', all_tags)

## 8. RoBERTa tokenizer (with `add_prefix_space=True`) + label alignment

RoBERTa uses byte-level BPE. When called with `is_split_into_words=True`, the tokenizer needs `add_prefix_space=True` so it knows each pre-split word should be treated as if preceded by a space. Without this, the first subword of every word after position 0 is tokenized as a "BPE-internal" piece rather than a "start-of-word" piece, and entity F1 silently collapses by ~10 absolute points.

The sanity-check cell below prints what BPE produces on a sample sentence. Every non-special token should start with `Ġ` (RoBERTa's space-prefix marker).

In [ ]:
bert_version = 'roberta-base'
tokenizer = AutoTokenizer.from_pretrained(bert_version, add_prefix_space=True)

# Tokenization sanity check — every non-special token below should start with `Ġ`
# (RoBERTa's space-prefix marker). If any non-first word's first subword lacks
# `Ġ`, `add_prefix_space=True` was forgotten and metrics will silently collapse.
_sample = ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
_enc = tokenizer(_sample, is_split_into_words=True)
_token_strings = tokenizer.convert_ids_to_tokens(_enc['input_ids'])
print('token strings:', _token_strings)
print('word_ids:    ', _enc.word_ids())
assert all('Ġ' in t or t in {'<s>', '</s>'} or t.startswith('▁') for t in _token_strings if not t.startswith('<')), \
    'TOKENIZER MISCONFIGURED: missing `Ġ` prefix — `add_prefix_space=True` was forgotten.'

def align_label(tokens, labels):
    word_ids = tokens.word_ids()
    previous_word_idx = None
    label_ids = []
    for word_idx in word_ids:
        if word_idx is None:
            label_ids.append(-100)
        elif word_idx != previous_word_idx:
            label_ids.append(label2id.get(labels[word_idx], -100))
        else:
            label_ids.append(-100)
        previous_word_idx = word_idx
    return label_ids

def encode(sentence):
    encodings = tokenizer(
        sentence['tokens'],
        truncation=True,
        padding='max_length',
        is_split_into_words=True,
        return_tensors='pt',
    )
    labels = align_label(encodings, sentence['ner_tags'])
    return {
        'input_ids': encodings['input_ids'].squeeze(0),
        'attention_mask': encodings['attention_mask'].squeeze(0),
        'labels': torch.tensor(labels, dtype=torch.long),
    }

print('encoding data')
train_dataset_raw = [encode(s) for s in train_sentences]
valid_dataset_raw = [encode(s) for s in valid_sentences]
test_dataset_raw = [encode(s) for s in test_sentences]

class InputDataset(torch.utils.data.Dataset):
    def __init__(self, data): self.data = data
    def __len__(self): return len(self.data)
    def __getitem__(self, idx): return self.data[idx]

train_dataset = InputDataset(train_dataset_raw)
valid_dataset = InputDataset(valid_dataset_raw)
test_dataset = InputDataset(test_dataset_raw)

## 9. Evaluation helpers — full 4 metrics (unchanged from Q1)

In [ ]:
def EvaluateModel(model, data_loader):
    model.eval()
    Y_actual_flat, Y_preds_flat = [], []
    y_true_tags, y_pred_tags = [], []
    with torch.no_grad():
        for batch in tqdm(data_loader, desc='Evaluating'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            preds = torch.argmax(outputs.logits, dim=-1)
            for idx in range(batch['labels'].size(0)):
                true_values_all = batch['labels'][idx]
                mask = (true_values_all != -100)
                true_values = true_values_all[mask]
                pred_values = preds[idx][mask]
                Y_actual_flat.append(true_values)
                Y_preds_flat.append(pred_values)
                y_true_tags.append([id2label[i] for i in true_values.tolist()])
                y_pred_tags.append([id2label[i] for i in pred_values.tolist()])
    Y_actual_flat = torch.cat(Y_actual_flat).detach().cpu().numpy()
    Y_preds_flat = torch.cat(Y_preds_flat).detach().cpu().numpy()
    return Y_actual_flat, Y_preds_flat, y_true_tags, y_pred_tags


def report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name):
    print(f'\n=== {split_name} — Token-level metrics ===')
    print('Accuracy          : {:.3f}'.format(accuracy_score(Y_actual, Y_preds)))
    print('Balanced accuracy : {:.3f}'.format(balanced_accuracy_score(Y_actual, Y_preds)))
    print(f'\n=== {split_name} — Entity-level metrics (seqeval) ===')
    print('Precision : {:.3f}'.format(seqeval_precision(y_true_tags, y_pred_tags)))
    print('Recall    : {:.3f}'.format(seqeval_recall(y_true_tags, y_pred_tags)))
    print('F1        : {:.3f}'.format(seqeval_f1(y_true_tags, y_pred_tags)))

## 10. Seed control + git SHA helper

In [ ]:
def set_seeds(seed: int) -> None:
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    if torch.backends.mps.is_available():
        torch.mps.manual_seed(seed)

def _git_sha():
    try:
        return subprocess.check_output(['git', 'rev-parse', 'HEAD'], text=True, stderr=subprocess.DEVNULL).strip()
    except Exception:
        return None

GIT_SHA = _git_sha()
PER_SEED_RESULTS = []

## 11. The Q8 change — same Q1 training loop, RoBERTa as the encoder

The full Q8 modification list:
1. **Import** `RobertaForTokenClassification` (instead of `BertForTokenClassification`).
2. **Model identifier** `roberta-base` (instead of `bert-base-uncased`).
3. **Tokenizer** `AutoTokenizer.from_pretrained(name, add_prefix_space=True)` — critical, see §8.
4. **Model class** `RobertaForTokenClassification.from_pretrained(...)`.

Everything else (NER task, 9 BIO classes, EPOCHS=3, BATCH_SIZE=8, LR=1e-5, AdamW, SEEDS=[42,43,44], align_label, eval helpers) is byte-identical to Q1.

In [ ]:
for run_index, seed in enumerate(SEEDS):
    print(f'\n{"#" * 60}\n# Q8 run {run_index + 1}/{len(SEEDS)} — seed={seed}\n{"#" * 60}')

    set_seeds(seed)

    # fresh model + head per seed
    print('initializing the model')
    # Q8 CHANGE: RobertaForTokenClassification instead of BertForTokenClassification
    model = RobertaForTokenClassification.from_pretrained(
        bert_version,
        num_labels=num_labels,
        id2label=id2label,
        label2id=label2id,
    )

    model = model.to(device)
    optimizer = optim.AdamW(params=model.parameters(), lr=LR)

    train_generator = torch.Generator().manual_seed(seed)
    train_loader = torch.utils.data.DataLoader(
        train_dataset, batch_size=BATCH_SIZE, shuffle=True, generator=train_generator,
    )
    valid_loader = torch.utils.data.DataLoader(valid_dataset, batch_size=BATCH_SIZE)
    test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=BATCH_SIZE)

    print('training the model')
    train_start = time.perf_counter()
    for epoch in range(EPOCHS):
        model.train()
        print(f'epoch {epoch + 1}/{EPOCHS}')
        for batch in tqdm(train_loader, desc=f'Training epoch {epoch + 1}'):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            loss = outputs.loss
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
        Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, valid_loader)
        report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags,
                       split_name=f'Validation (epoch {epoch + 1})')
    training_seconds = time.perf_counter() - train_start

    print(f'\napplying the model to the test set (seed={seed})')
    Y_actual, Y_preds, y_true_tags, y_pred_tags = EvaluateModel(model, test_loader)
    report_metrics(Y_actual, Y_preds, y_true_tags, y_pred_tags, split_name=f'Test (seed={seed})')

    label_ids_sorted = list(range(num_labels))
    target_names = [id2label[i] for i in label_ids_sorted]
    print(f'\n=== Test (seed={seed}) — Token-level classification report (sklearn) ===')
    print(classification_report(Y_actual, Y_preds, labels=label_ids_sorted,
                                target_names=target_names, zero_division=0))
    print(f'=== Test (seed={seed}) — Entity-level classification report (seqeval) ===')
    print(seqeval_report(y_true_tags, y_pred_tags, digits=3, zero_division=0))

    # NOTE on JSON shape: top-level `script` matches Q1's key for aggregator parity.
    # `notebook` is ALSO included so the JSON traces back to the Colab runtime.
    # `model` field changes from "bert-base-uncased" → "roberta-base" — this is
    # what an aggregator script keys on to distinguish Q1 vs Q8.
    metrics = {
        'question': 'Q8',
        'script': 'src/q8_roberta.py',
        'notebook': 'notebooks/08_q8_roberta.ipynb',
        'model': bert_version,
        'task': 'ner',
        'seed': seed,
        'run_index': run_index,
        'epochs': EPOCHS,
        'batch_size': BATCH_SIZE,
        'lr': LR,
        'training_seconds': training_seconds,
        'test': {
            'token_micro_accuracy': float(accuracy_score(Y_actual, Y_preds)),
            'token_macro_accuracy': float(balanced_accuracy_score(Y_actual, Y_preds)),
            'entity_micro_f1': float(seqeval_f1(y_true_tags, y_pred_tags)),
            'entity_macro_f1': float(seqeval_f1(y_true_tags, y_pred_tags, average='macro')),
        },
        'device': str(device),
        'git_commit': GIT_SHA,
    }
    out_path = RESULTS_DIR / f'seed_{seed}.json'
    out_path.write_text(json.dumps(metrics, indent=2) + '\n')
    print(f'\nsaved metrics: {out_path}')
    PER_SEED_RESULTS.append(metrics)

    del model, optimizer, train_loader, valid_loader, test_loader
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    if torch.backends.mps.is_available(): torch.mps.empty_cache()

## 12. Summary across seeds

In [ ]:
print(f'\n{"#" * 60}\n# Q8 — summary across {len(SEEDS)} seeds\n{"#" * 60}')
metric_keys = ['token_micro_accuracy', 'token_macro_accuracy', 'entity_micro_f1', 'entity_macro_f1']
print(f"{'metric':30s}  {'mean':>10s}  {'stdev':>10s}  values")
for k in metric_keys:
    vals = [r['test'][k] for r in PER_SEED_RESULTS]
    m = statistics.mean(vals)
    s = statistics.stdev(vals) if len(vals) > 1 else 0.0
    vals_str = ', '.join(f'{v:.4f}' for v in vals)
    print(f'{k:30s}  {m:>10.4f}  {s:>10.4f}  [{vals_str}]')
times = [r['training_seconds'] for r in PER_SEED_RESULTS]
print(f"{'training_seconds':30s}  {statistics.mean(times):>10.1f}  {statistics.stdev(times) if len(times) > 1 else 0.0:>10.1f}  [{', '.join(f'{t:.1f}' for t in times)}]")

## 13. Download results (skip if Drive mount above ran)

If you ran cell §5b, results are already in Drive — no download needed. Otherwise each `files.download()` call opens a browser-native save dialog; drop the files into `results/q8/` in your local repo and commit.

In [ ]:
from google.colab import files
for f in sorted(RESULTS_DIR.glob('seed_*.json')):
    print(f'downloading {f.name}')
    files.download(str(f))